## Тестирование жадных политик

In [1]:
import torch
from env import FJSPTEnv
from pathlib import Path
import pandas as pd
from greedy_policies import GreedyPolicy
from rl4co.envs import ENV_REGISTRY

ENV_REGISTRY["fjspt"] = FJSPTEnv

datasets_params = [
    ("1_Deroussi_and_Norre", "datasets/1_Deroussi_and_Norre/proc_data", 1),
    ("2_Ham_edata", "datasets/2_Ham/edata", 1),
    ("2_Ham_rdata", "datasets/2_Ham/rdata", 1),
    ("2_Ham_sdata", "datasets/2_Ham/sdata", 1),
    ("2_Ham_vdata", "datasets/2_Ham/vdata", 1),
    ("3_Homayouni_and_Fontes-Brandimarte", "datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data", 1),
    ("4_Homayouni_and_Fontes-Fattahi", "datasets/4_Homayouni_and_Fontes-Fattahi/proc_data", 0),
    ("5_Dauzere", "datasets/5_Dauzere/proc_data", 1),
]

results = {}
device = torch.device("cpu")
strategies = ["FIFO", "MOPNR", "SPT", "MWKR"]
policies = {
    s: GreedyPolicy(policy_type=s).to(device).eval()
    for s in strategies
}

for dataset_name, proc_path, ma_indexing in datasets_params:
    print("-" * 20)
    print(proc_path)
    path = Path(proc_path)
    rows = []
    
    for file in path.iterdir():
        if not file.is_file():
            continue
        print(file)
        row = {"file": str(file).split("/")[-1]}
            
        for strategy in strategies:
            print(strategy)
            policy = policies[strategy]
  
            # Окружение для тестирования модели с помощью подготовленных датасетов
            env = FJSPTEnv(
                generator_params={
                  # датасеты с данными FJSP
                  "proc_file_path": file,
                  # датасет с временем транспортировки
                  "trucks_file_path": Path(proc_path).parent / "trucks_data",
                  "ma_indexing": ma_indexing,
                },
            )
            
            td = env.reset(batch_size=[1]).to(device)
            
            while not td["done"].all():
                out = policy(td)
                td["action"] = out[0].argmax(dim=1)
                td = env.step(td)["next"]
                
            reward = td["machine_finish_times"].masked_fill(td["pad_mask"], -torch.inf).max(1).values
            row[strategy] = reward.item()
            
        rows.append(row)

    df = pd.DataFrame(rows).sort_values("file")
    save_path = f"greedy_results/{dataset_name}.csv"
    df.to_csv(save_path, index=False)
    print(f"Saved to {save_path}")

--------------------
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt10.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt01.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt05.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt02.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt06.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt08.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt07.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt03.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt09.txt
FIFO
MOPNR
SPT
MWKR
../datasets/3_Homayouni_and_Fontes-Brandimarte/proc_data/mkt04.txt
FIFO
MOPNR
SPT
MWKR
Saved to greedy_results/3_Homayouni_and_Fontes-Brand

Усреднение makespan $С_{max}$ для датасета Ham

In [4]:
import pandas as pd

files = ["edata", "rdata", "sdata", "vdata"]
results = []
for HUdata in files:
    df = pd.read_csv(f"greedy_results/2_Ham_{HUdata}.csv", sep=',', index_col=0)
    means = df[['FIFO', 'MOPNR', 'SPT', 'MWKR']].mean().round(1)
    means['HUdata'] = HUdata
    means = means[['HUdata', 'FIFO', 'MOPNR', 'SPT', 'MWKR']]
    results.append(means)


result_df = pd.DataFrame(results)
result_df.to_csv("greedy_results/2_Ham.csv", index=False)
result_df

,HUdata,FIFO,MOPNR,SPT,MWKR
0,edata,6476.6,7899.1,7004.1,7615.0
1,rdata,5980.1,7226.2,7181.0,7288.7
2,sdata,6739.5,8466.1,7080.6,6949.8
3,vdata,5594.5,7825.8,7853.7,7999.1
